In [1]:
import pandas as pd
from rdkit import Chem, DataStructs


df = pd.read_csv('test_results/test.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'test_results/test.csv'

In [ ]:
def get_similarity(pred, true_metabolites):
    """
    get the similarity between the prediction and the true metabolites
    : param pred: a list of prediction smiles
    : param true_metabolites: a list of true metabolites smiles
    : return: the maxinum similarity between the prediction and the true metabolites
    """
    max_similarity = -5
    for metabolite in true_metabolites:
        metabolite_mol = Chem.MolFromSmiles(metabolite)
        metabolite_fgp = Chem.RDKFingerprint(metabolite_mol)
        prediction_mol = Chem.MolFromSmiles(pred)
        prediction_fgp = Chem.RDKFingerprint(prediction_mol)

        sim = DataStructs.FingerprintSimilarity(metabolite_fgp, prediction_fgp)
        if sim > max_similarity:
            max_similarity = sim
            
    return max_similarity

In [ ]:
corr = 0 
at_least_one, at_least_half, all_metabolites = 0, 0, 0
len_preds, len_corr_preds, len_actual_metabolites, invaild = 0, 0, 0, 0


for i in range(len(df)):
    substrate = df.iloc[i]['substrate']
    true_metabolites = df.iloc[i]['metabolite'].strip().split('|')
    original_preds = df.iloc[i]['all_predictions'].strip().split('|')
    original_preds = list(set(original_preds))
    len_preds += len(original_preds)

    corr_preds = []
    for pred in original_preds:
        try:
            pred = Chem.MolToSmiles(Chem.MolFromSmiles(pred))
        except:
            invaild += 1
            continue

    
        # if pred in true_metabolites:
        #     corr_preds.append(pred)

        if get_similarity(pred, true_metabolites) == 1:
            corr_preds.append(pred)

    if len(corr_preds) > 0:
        at_least_one += 1
    if len(corr_preds) >= len(true_metabolites)/2:
        at_least_half += 1
    if len(corr_preds) == len(true_metabolites):
        all_metabolites += 1
        print(substrate)
        print(corr_preds)

    len_corr_preds += len(corr_preds)
    len_actual_metabolites += len(true_metabolites)

print('at least one: {:.2%}'.format(at_least_one/len(df)))
print('at least half: {:.2%}'.format(at_least_half/len(df)))
print('all metabolites: {:.2%}'.format(all_metabolites/len(df)))
print('precision: {:.2%}'.format(len_corr_preds/len_preds))
print('recall: {:.2%}'.format(len_corr_preds/len_actual_metabolites))
print('invalid: {:.2%}'.format(invaild/len_preds))

